In [1]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from pyvis.network import Network
import os
import matplotlib.colors as mcolors

In [2]:
# this folder path needs to be changed for each computer
folder_path='/Users/pipe/emergence_of_coordinated_cell_division_during_the_evolution_of_multicellularity/Data/network_files'
os.chdir(folder_path)

In [3]:
# Loading the graphs
petite_net=nx.read_graphml('petite_growth_200_cells_1_14nov2024.graphml')
grande_net=nx.read_graphml('grande_growth_200_cells_1_3dec2024.graphml')


In [4]:

# Function to find the diameter path
def find_diameter_path(graph):
    paths = dict(nx.all_pairs_shortest_path(graph))
    diameter_path = max((path for paths_from_node in paths.values() for path in paths_from_node.values()), key=len)
    return diameter_path

# Function to create and save network visualization
def create_network_visualization(graph, filename):
    nt = Network('1000px', '1000px', notebook=True)
    nt.from_nx(graph)
    
    # Find the diameter path
    diameter_path = find_diameter_path(graph)
    diameter_edges = list(zip(diameter_path[:-1], diameter_path[1:]))
    
    # Identify nodes with 2 or more degree-1 neighbors
    nodes_with_degree1_neighbors = [node for node in graph.nodes() if sum(1 for neighbor in graph.neighbors(node) if graph.degree(neighbor) == 1) >= 2]
    
    for node in nt.nodes:
        node_id = node['id']
        node['size'] = 25
        node['label'] = ''
        
        if graph.degree(node_id) == 1 and any(neighbor in nodes_with_degree1_neighbors for neighbor in graph.neighbors(node_id)):
            node['color'] = 'red'
        elif node_id in nodes_with_degree1_neighbors:
            node['color'] = 'red'
        else:
            node['color'] = 'blue'
    
    for edge in nt.edges:
        if (edge['from'], edge['to']) in diameter_edges or (edge['to'], edge['from']) in diameter_edges:
            edge['color'] = 'black'
            edge['width'] = 12  # Thicker line for diameter path
        elif (graph.degree(edge['from']) == 1 and edge['to'] in nodes_with_degree1_neighbors) or \
             (graph.degree(edge['to']) == 1 and edge['from'] in nodes_with_degree1_neighbors):
            edge['color'] = 'red'
            edge['width'] = 7
        else:
            edge['color'] = 'blue'
            edge['width'] = 7
    
    nt.toggle_hide_edges_on_drag(True)
    nt.show_buttons(filter_=['physics'])
    nt.set_edge_smooth('dynamic')
    nt.save_graph(filename)

# Create visualizations for both networks
create_network_visualization(petite_net, 'petite_highlight_undivided_cells.html')
create_network_visualization(grande_net, 'grande_highlight_undivided_cells.html')